### Imports

In [2]:
import random

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from collections import Counter

### Tiny training dataset

In [3]:
TRAIN_TEXTS = [
    "please deliver the parcel tomorrow",
    "please send the package tomorrow",
    "the parcel should arrive tomorrow",
    "the package needs to be delivered tomorrow",
    "send the parcel to the customer",
    "deliver the package to the customer",
    "the customer will receive the parcel",
    "the customer should receive the package",
    "please deliver the shipment today",
    "please send the shipment today",
    "the shipment should arrive today",
    "the package should arrive today",
    "we need to send the parcel",
    "we need to deliver the package",
    "the parcel has to be delivered",
    "the package has to be sent",
    "please send the documents tomorrow",
    "please deliver the documents tomorrow",
    "the documents should arrive tomorrow",
    "send the documents to the customer",
]

### Tokenizer

In [4]:
PAD = "<pad>"
BOS = "<bos>"
EOS = "<eos>"
UNK = "<unk>"

In [5]:
def tokenize(text):
    return text.lower().split()


def build_vocab(texts, min_freq=1):
    counts = Counter(tok for text in texts for tok in tokenize(text))

    vocab = [PAD, BOS, EOS, UNK]
    vocab += sorted(tok for tok, n in counts.items() if n >= min_freq)

    stoi = {w: i for i, w in enumerate(vocab)}
    itos = {i: w for w, i in stoi.items()}
    return stoi, itos

In [6]:
STOI, ITOS = build_vocab(TRAIN_TEXTS)

### Text encoding

In [7]:
def encode_text(text, max_len=12):
    ids = [STOI[BOS]]
    ids += [STOI.get(tok, STOI[UNK]) for tok in tokenize(text)]
    ids += [STOI[EOS]]

    ids = ids[:max_len]

    if len(ids) < max_len:
        ids += [STOI[PAD]] * (max_len - len(ids))

    return torch.tensor(ids, dtype=torch.long)

### Dataset

In [8]:
class TextDataset(Dataset):
    def __init__(self, texts, max_len=12):
        self.data = [encode_text(x, max_len) for x in texts]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

### Model

In [18]:
class DPVAE(nn.Module):
    def __init__(
        self,
        vocab_size,
        emb_dim=64,
        hidden_dim=96,
        latent_dim=32,
        noise_std=0.35,
        latent_radius=1.0,
        pad_id=0,
    ):
        super().__init__()

        self.latent_dim = latent_dim
        self.noise_std = noise_std
        self.latent_radius = latent_radius
        self.pad_id = pad_id

        self.embedding = nn.Embedding(
            vocab_size,
            emb_dim,
            padding_idx=pad_id,
        )

        # Small GRU encoder.
        self.encoder = nn.GRU(
            emb_dim,
            hidden_dim,
            batch_first=True,
        )

        self.to_mu = nn.Linear(hidden_dim, latent_dim)
        self.to_logvar = nn.Linear(hidden_dim, latent_dim)

        # Decoder receives the latent vector at every time step.
        self.decoder = nn.GRU(
            emb_dim + latent_dim,
            hidden_dim,
            batch_first=True,
        )

        self.output = nn.Linear(hidden_dim, vocab_size)

    def encode(self, x):
        emb = self.embedding(x)
        _, h = self.encoder(emb)

        h = h[-1]

        # Bounded latent mean.
        #
        # This is the important DP-VAE step:
        #
        # mu* = R * tanh(mu)
        #
        # so the latent representation cannot grow without bound.
        mu = self.latent_radius * torch.tanh(self.to_mu(h))

        logvar = self.to_logvar(h).clamp(-6.0, 2.0)

        return mu, logvar

    def add_dp_noise(self, mu):
        # Input-independent Gaussian noise.
        noise = torch.randn_like(mu) * self.noise_std
        return mu + noise

    def decode(self, z, x_in):
        """
        Teacher-forced decoder during training.
        """
        emb = self.embedding(x_in)

        z_seq = z.unsqueeze(1).expand(-1, emb.size(1), -1)

        decoder_input = torch.cat([emb, z_seq], dim=-1)

        h, _ = self.decoder(decoder_input)
        logits = self.output(h)

        return logits

    def forward(self, x):
        mu, logvar = self.encode(x)

        # Standard VAE reparameterization.
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z_vae = mu + eps * std

        # DP version: use the bounded mean plus fixed Gaussian noise.
        z_dp = self.add_dp_noise(mu)

        # Decoder input excludes the final token.
        logits = self.decode(z_dp, x[:, :-1])

        return logits, mu, logvar

    @torch.no_grad()
    def generate(
        self,
        text,
        max_new_tokens=15,
        temperature=0.8,
        noise_scale=None,
        deterministic=False,
    ):
        """
        Encode an input sentence and generate a semantically related
        realization from a noisy latent representation.

        noise_scale:
            1.0 -> training noise_std
            0.0 -> no DP noise (useful for debugging)
            >1  -> stronger obfuscation, usually lower semantic fidelity

        deterministic=True:
            disables sampling in the decoder; useful for debugging.
        """
        self.eval()

        if noise_scale is None:
            noise_scale = 1.0

        x = encode_text(text).unsqueeze(0)

        mu, _ = self.encode(x)

        if noise_scale == 0:
            z = mu
        else:
            z = mu + torch.randn_like(mu) * self.noise_std * noise_scale

        current = torch.tensor(
            [[STOI[BOS]]],
            dtype=torch.long,
            device=x.device,
        )

        result = []

        for _ in range(max_new_tokens):
            logits = self.decode(z, current)

            next_logits = logits[:, -1, :] / max(temperature, 1e-5)

            # Never generate padding / BOS.
            next_logits[:, STOI[PAD]] = -float("inf")
            next_logits[:, STOI[BOS]] = -float("inf")

            if deterministic:
                next_token = next_logits.argmax(dim=-1, keepdim=True)
            else:
                probs = F.softmax(next_logits, dim=-1)
                next_token = torch.multinomial(probs, 1)

            token_id = next_token.item()

            if token_id == STOI[EOS]:
                break

            result.append(ITOS[token_id])
            current = torch.cat([current, next_token], dim=1)

        return " ".join(result)

### Training loop

In [19]:
def train_model(
    epochs=250,
    batch_size=8,
    lr=2e-3,
    beta=0.01,
    noise_std=0.35,
    device="cpu",
):
    dataset = TextDataset(TRAIN_TEXTS)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    model = DPVAE(
        vocab_size=len(STOI),
        noise_std=noise_std,
        pad_id=STOI[PAD],
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(1, epochs + 1):
        model.train()

        total_loss = 0.0
        total_recon = 0.0
        total_kl = 0.0

        for x in loader:
            x = x.to(device)

            logits, mu, logvar = model(x)

            # Target is the sequence shifted by one token.
            target = x[:, 1:]

            recon_loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                target.reshape(-1),
                ignore_index=STOI[PAD],
            )

            # Standard VAE KL term.
            kl = -0.5 * torch.mean(
                1 + logvar - mu.pow(2) - logvar.exp()
            )

            loss = recon_loss + beta * kl

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_loss += loss.item()
            total_recon += recon_loss.item()
            total_kl += kl.item()

        if epoch % 25 == 0 or epoch == 1:
            n = len(loader)
            print(
                f"epoch={epoch:03d} "
                f"loss={total_loss/n:.4f} "
                f"recon={total_recon/n:.4f} "
                f"kl={total_kl/n:.4f}"
            )

    return model

### Train the model

In [30]:
random.seed(42)
torch.manual_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"

model = train_model(
    epochs=1000,
    noise_std=0.35,
    device=device,
)

epoch=001 loss=3.2177 recon=3.2177 kl=0.0027
epoch=025 loss=0.6279 recon=0.6243 kl=0.3675
epoch=050 loss=0.2277 recon=0.2238 kl=0.3967
epoch=075 loss=0.0798 recon=0.0756 kl=0.4190
epoch=100 loss=0.0345 recon=0.0302 kl=0.4319
epoch=125 loss=0.0215 recon=0.0172 kl=0.4314
epoch=150 loss=0.0141 recon=0.0097 kl=0.4342
epoch=175 loss=0.0213 recon=0.0170 kl=0.4304
epoch=200 loss=0.0103 recon=0.0059 kl=0.4328
epoch=225 loss=0.0100 recon=0.0057 kl=0.4304
epoch=250 loss=0.0071 recon=0.0029 kl=0.4232
epoch=275 loss=0.0066 recon=0.0023 kl=0.4231
epoch=300 loss=0.0063 recon=0.0022 kl=0.4092
epoch=325 loss=0.0062 recon=0.0021 kl=0.4109
epoch=350 loss=0.0056 recon=0.0016 kl=0.3992
epoch=375 loss=0.0053 recon=0.0014 kl=0.3884
epoch=400 loss=0.0050 recon=0.0013 kl=0.3717
epoch=425 loss=0.0078 recon=0.0039 kl=0.3893
epoch=450 loss=0.0059 recon=0.0019 kl=0.4028
epoch=475 loss=0.0053 recon=0.0013 kl=0.3996
epoch=500 loss=0.0054 recon=0.0013 kl=0.4155
epoch=525 loss=0.0051 recon=0.0010 kl=0.4056
epoch=550 

### Generate text

In [31]:
examples = [
    "please deliver the parcel tomorrow",
    "the package should arrive today",
    "send the documents to the customer",
    "we need to deliver the package",
]

print("\nGenerated text:\n")

for text in examples:
    print("INPUT :", text)
    print(
        "OUTPUT:",
        model.generate(
            text,
            noise_scale=1.0,
            temperature=0.8,
        ),
    )

    print(
        "NO NOISE:",
        model.generate(
            text,
            noise_scale=0.0,
            deterministic=True,
        ),
    )
    print()


Generated text:

INPUT : please deliver the parcel tomorrow
OUTPUT: please deliver the parcel tomorrow
NO NOISE: please deliver the parcel tomorrow

INPUT : the package should arrive today
OUTPUT: the package should arrive today
NO NOISE: the package should arrive today

INPUT : send the documents to the customer
OUTPUT: send the documents to the customer
NO NOISE: send the documents to the customer

INPUT : we need to deliver the package
OUTPUT: we need to deliver the package
NO NOISE: we need to deliver the package

